In [1]:
import os 
os.chdir("../")
%pwd

'd:\\Programming\\ML\\End-to-End\\End-to-End-TelcoChurn'

In [2]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/youssefBedeer/End-to-End-TelcoChurn.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="youssefBedeer"
os.environ["MLFLOW_TRACKING_PASSWORD"]="1310a8b0d5a018b5a0f0e66fc38a5f9b5fbb6936"

In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    X_test_path: Path
    y_test_path: Path
    model_path: Path
    best_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str
    THRESHOLD : float

In [4]:
os.getenv("MLFLOW_TRACKING_URI")

'https://dagshub.com/youssefBedeer/End-to-End-TelcoChurn.mlflow'

In [5]:
from src.constants import *
from src.utils import read_yaml, create_directories, save_json

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        best_params_path= BEST_PARAMS_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.best_params= read_yaml(best_params_path)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.best_params
        schema =  self.schema.TARGET_COLUMN
        

        create_directories([config.root_dir])

        return ModelEvaluationConfig(
            root_dir =  Path(config.root_dir),
            X_test_path = Path(config.X_test_path),
            y_test_path =  Path(config.y_test_path),
            model_path = Path(config.model_path),
            best_params = params,
            metric_file_name = Path(config.metric_file_name),
            target_column = schema.name,
            mlflow_uri=os.getenv("MLFLOW_TRACKING_URI"),
            THRESHOLD = config.THRESHOLD,
        )



In [8]:
import contextlib
import os
import pandas as pd
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib
from src import CustomException
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from pathlib import Path
from src.utils import save_json  # adjust import to your project structure
from scipy import sparse
from src import logging


class ModelEvaluation:
    def __init__(self, config):
        self.config = config

    def eval_metrics(self, actual, pred, pred_proba=None):
        """Compute classification metrics"""
        accuracy = accuracy_score(actual, pred)
        precision = precision_score(actual, pred, zero_division=0, pos_label=1)
        recall = recall_score(actual, pred, zero_division=0, pos_label=1)
        f1 = f1_score(actual, pred,  pos_label=1)

        metrics = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

        # Add ROC-AUC if probabilities are available and the problem is binary
        if pred_proba is not None and len(np.unique(actual)) == 2:
            with contextlib.suppress(Exception):
                auc = roc_auc_score(actual, pred_proba[:, 1])
                metrics["roc_auc"] = auc
        return metrics

    def log_into_mlflow(self):
        """Evaluate model on test data and log metrics to MLflow"""
        X_test_arr = sparse.load_npz(self.config.X_test_path)
        y_test = pd.read_csv(self.config.y_test_path).values.ravel()
        model = joblib.load(self.config.model_path)


        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            proba = model.predict_proba(X_test_arr)[:, 1]
            predicted_classes = (proba >= self.config.THRESHOLD).astype(int)

            # If model supports probability prediction
            pred_proba = None
            if hasattr(model, "predict_proba"):
                try:
                    pred_proba = model.predict_proba(X_test_arr)
                except Exception:
                    pass

            metrics = self.eval_metrics(y_test, predicted_classes)

            # Save metrics locally
            save_json(path=Path(self.config.metric_file_name), data=metrics)

            # Log parameters and metrics to MLflow
            mlflow.log_params(self.config.best_params)
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(metric_name, metric_value)

            # log model 
            # mlflow.xgboost.log_model(model, name="xgboost")




In [9]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise CustomException(e)

[2025-10-19 00:32:36,437] [INFO] [root:read_yaml:16] - reading the content of 'config\config.yaml'
[2025-10-19 00:32:36,445] [INFO] [root:read_yaml:16] - reading the content of 'params.yaml'
[2025-10-19 00:32:36,449] [INFO] [root:read_yaml:16] - reading the content of 'best_params.yaml'
[2025-10-19 00:32:36,456] [INFO] [root:read_yaml:16] - reading the content of 'schema.yaml'
[2025-10-19 00:32:36,459] [INFO] [root:create_directories:39] - created directory at: artifacts
[2025-10-19 00:32:36,460] [INFO] [root:create_directories:39] - created directory at: artifacts/data_evaluation
[2025-10-19 00:32:37,323] [INFO] [root:save_json:51] - JSON file saved at: artifacts\model_evaluation\scores.json
🏃 View run dashing-snail-689 at: https://dagshub.com/youssefBedeer/End-to-End-TelcoChurn.mlflow/#/experiments/0/runs/8de4e40de61c459daf790eba01a87fd4
🧪 View experiment at: https://dagshub.com/youssefBedeer/End-to-End-TelcoChurn.mlflow/#/experiments/0


In [ ]:
import joblib 
model= joblib.load("artifacts/model_trainer/model.joblib")
params = model.get_params()
print(params)

In [ ]:
import joblib
import pandas as pd
from scipy import sparse
from sklearn.metrics import recall_score

# Load model
model = joblib.load(r"D:\Programming\ML\End-to-End\End-to-End-TelcoChurn\artifacts\model_trainer\model.joblib")

# Load data
X_test_arr = sparse.load_npz("artifacts/data_transformation/X_test.npz")
y_test = pd.read_csv("artifacts/data_transformation/y_test.csv").values.ravel()

# Use the same threshold as during training
THRESHOLD = 0.3  # or read from YAML config

# Predict
proba = model.predict_proba(X_test_arr)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

# Evaluate
recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
print(f"Recall: {recall:.4f}")
